# 06 – Model Evaluation

**Goal:** Evaluate both models on the same held-out test records using confusion matrices, precision, recall and F1-score.

For this project, recall is important because missed suspicious examples matter. Precision also matters because too many false alarms can burden investigators.

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "dataset"
MODEL_DIR = PROJECT_ROOT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

FEATURES = joblib.load(MODEL_DIR / "feature_names.joblib")
rf_model = joblib.load(MODEL_DIR / "random_forest_model.joblib")
iso_model = joblib.load(MODEL_DIR / "isolation_forest_model.joblib")

test_df = pd.read_csv(MODEL_DIR / "test_data.csv")
X_test = test_df[FEATURES]
y_test = test_df["threat_label"].astype(int)

print("Test set shape:", X_test.shape)

In [ ]:
# Random Forest evaluation
rf_pred = rf_model.predict(X_test)

print("Random Forest accuracy:", round(accuracy_score(y_test, rf_pred), 4))
print("Random Forest precision:", round(precision_score(y_test, rf_pred, zero_division=0), 4))
print("Random Forest recall:", round(recall_score(y_test, rf_pred, zero_division=0), 4))
print("Random Forest F1:", round(f1_score(y_test, rf_pred, zero_division=0), 4))

print("\nClassification report:")
print(classification_report(y_test, rf_pred, zero_division=0))

In [ ]:
# Random Forest confusion matrix
cm_rf = confusion_matrix(y_test, rf_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm_rf, annot=True, fmt="d", cmap="Blues")
plt.title("Random Forest Confusion Matrix")
plt.xlabel("Predicted label")
plt.ylabel("Actual label")
plt.tight_layout()
plt.show()

In [ ]:
# Isolation Forest returns 1 for inliers and -1 for outliers.
# Convert this to our project convention: 0 = Normal, 1 = Suspicious.
iso_raw = iso_model.predict(X_test)
iso_pred = (iso_raw == -1).astype(int)

print("Isolation Forest accuracy:", round(accuracy_score(y_test, iso_pred), 4))
print("Isolation Forest precision:", round(precision_score(y_test, iso_pred, zero_division=0), 4))
print("Isolation Forest recall:", round(recall_score(y_test, iso_pred, zero_division=0), 4))
print("Isolation Forest F1:", round(f1_score(y_test, iso_pred, zero_division=0), 4))

print("\nClassification report:")
print(classification_report(y_test, iso_pred, zero_division=0))

In [ ]:
# Isolation Forest confusion matrix
cm_iso = confusion_matrix(y_test, iso_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm_iso, annot=True, fmt="d", cmap="Oranges")
plt.title("Isolation Forest Confusion Matrix")
plt.xlabel("Predicted label")
plt.ylabel("Actual label")
plt.tight_layout()
plt.show()

In [ ]:
# Compare model metrics in one table
comparison = pd.DataFrame({
    "Model": ["Random Forest", "Isolation Forest"],
    "Accuracy": [
        accuracy_score(y_test, rf_pred),
        accuracy_score(y_test, iso_pred)
    ],
    "Precision": [
        precision_score(y_test, rf_pred, zero_division=0),
        precision_score(y_test, iso_pred, zero_division=0)
    ],
    "Recall": [
        recall_score(y_test, rf_pred, zero_division=0),
        recall_score(y_test, iso_pred, zero_division=0)
    ],
    "F1-score": [
        f1_score(y_test, rf_pred, zero_division=0),
        f1_score(y_test, iso_pred, zero_division=0)
    ]
})
comparison.round(3)

## Final interpretation

Do not claim that a high score on this simulated dataset proves real-world threat detection ability. The synthetic labels were created using activity patterns, so the model can learn those same patterns. A real evaluation requires suitable, independently sourced data, careful validation, and human review.